In [1]:
from ollama import AsyncClient

model='llama3.1:8b'
client = AsyncClient(host='http://localhost:11434')

In [ ]:
import re
import json

EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 


class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str, product_id_req: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""

    ret = {"product_id": product_id_req}

    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        return ret | {'error': f"Ответ не является корректным JSON: {error.msg}"}

    if not isinstance(data, dict):
        return ret | {'error': "На верхнем уровне должен находиться JSON-объект"}

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        }

    if unexpected_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        }

    product_id = data['product_id']
    if product_id != product_id_req:
        return ret | {'error': "Поле product_id не соответствует входным параметрам"}

    description = data["description"]

    if not isinstance(description, str):
        return ret | {'error': "Поле description должно быть строкой"}

    description = description.strip()

    if not description:
        return ret | {'error': "Поле description не должно быть пустым"}
      
    i = count_sentences(description)
    if i < 2 or i > 5:
        return ret | {'error': "Description must contain 2-5 sentences"}

    if len(description) < MIN_DESCRIPTION_LENGTH:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Поле description содержит больше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    if len(description) > MAX_DESCRIPTION_LENGTH:
        return ret | {'error': "Поле description содержит меньше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    pros = data['pros']
    if len(pros) < 2 or len(pros) > 5:
       return ret | {'error': "Поле pros must contain 2-5 values"}

    cons = data['cons']
    if len(cons) < 1 or len(cons) > 3:
       return ret | {'error': "Поле cons must contain 1-3 values"}
    
    tags = data['tags']
    if len(tags) < 3 or len(tags) > 8:
       return ret | {'error': "Поле tags must contain 3-8 values"}
    
    return ret

In [5]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)

class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [4]:
response_schema = CardRequest.model_json_schema()
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [ ]:
async def request_one(
    msg: list[dict], 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> tuple[str, int]:
  messages=[
    {"role": "system", "content": 
      """
        Generate a product card in Russian (utf-8) by example.
        Example:
        {
          "product_id": "P-1001",
          "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
          "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
          "cons": ["Быстро разряжаются на морозе"],
          "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
        } 
        "description": 2-5 sentences,
        return only JSON
      """
    } 
  ] + msg
  # print('>>> messages', messages)

  response = await client.chat(
      model=model,  
      messages=messages,
      options={
        'seed':               42,  
        'temperature':        params['temperature'],  
        'top_p':              params['top_p'],
        'max_tokens':         params['max_tokens'],   
        "top_k":              params['top_k'], 
        "repetition_penalty": params['repetition_penalty'],      
      },
      format=CardRequest.model_json_schema(),
  )

  content = response['message']['content']
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )

  output_tokens = response.eval_count
  
  return content, output_tokens

In [12]:
import time

async def worming_up():
  item = {"product_id": "P-0026", "title": "Смартфон Grafo G9", "category": "Электроника / Смартфоны", "attributes": {"бренд": "Grafo", "экран": "6.7\"", "память": "256 ГБ", "оперативная память": "8 ГБ", "аккумулятор": "6000 мА·ч", "основная камера": "50 Мп"}, "reviews": ["Пользуюсь пару недель. Камера снимает достойно даже вечером", "Брал для себя. Звонки чистые, собеседника слышно хорошо", "В целом Быстро заряжается, за час почти полный", "Заказывал на подарок. Работает шустро, приложения не тормозят", "Брал для себя. Камера мылит фото при недостатке света", "Пользуюсь пару недель. К вечеру заряд просаживается быстрее ожидаемого", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь каждый день. Экран яркий, на солнце всё видно", "Быстро заряжается, за час почти полный", "В руке лежит удобно, не скользит", "Динамик тихий, в шумном месте плохо слышно", "Работает шустро, приложения не тормозят", "В руке лежит удобно, не скользит", "Коротко: Работает шустро, приложения не тормозят", "Купил на замену старому. Обычный телефон на каждый день, без сюрпризов", "Брал для себя. Батарея держит полтора дня без подзарядки", "В целом Работает шустро, приложения не тормозят"]}

  row_dict = [{"role": "user", "content": f"""{item}"""}]

  started = time.perf_counter()
  resp, tokens = await request_one(row_dict)
  generation_time = time.perf_counter() - started

  print('>>> worming_up response, tokens, time:\n', resp, tokens, generation_time)

In [14]:
await worming_up()

>>> worming_up response, tokens, time:
 {
  "product_id": "P-0026",
  "description": "Смартфон Grafo G9 - идеальный выбор для тех, кто ценит качество и функциональность. Экран размером 6,7\" обеспечивает яркий и четкий изображение, а камера с разрешением 50 Мп позволяет снимать высококачественные фотографии даже в условиях недостаточного освещения. Батарея с емкостью 6000 мА·ч обеспечивает долгое время работы без подзарядки.",
  "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы", "Быстро заряжается", "В руке лежит удобно, не скользит"],
  "cons": ["Динамик тихий, в шумном месте плохо слышно"],
  "tags": ["Смартфон", "Grafo", "Экран 6.7\"", "Камера 50 Мп", "Батарея 6000 мА·ч"]
} 242 86.8853937999811


In [28]:
MAX_ATTEMPTS = 3 

async def generate_one(
    msg: list[dict], 
    product_id: str,
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350},
) -> tuple[str, int]:
  messages_history = msg.copy()

  counter = 0
  errors = 0
  for attempt in range(1, MAX_ATTEMPTS + 1):
    counter += 1
    content, tokens = await request_one(messages_history, params)

    error_dict = parse_and_validate(content, product_id)
    if not 'error' in error_dict:
      return content
    else:
      errors += 1
      print('>>> error: ', error_dict)
      print('>>> error content: ', content)

      if attempt == MAX_ATTEMPTS:
        return {"product_id": product_id, "error": "ResponseValidationError"}

      messages_history = messages_history + [
        {
          "role": "assistant",
          "content": content,
        },
        {
          "role": "user",
          "content": (
              "Предыдущий ответ не прошёл проверку. "
              f"Причина: {error_dict}. "
              "Исправь ответ и верни только JSON"
          ),
        }
      ]

In [16]:
# test generate_one
item = {
  "product_id": "P-0027", 
  "title": "Пылесос Klin Eco", 
  "category": "Бытовая техника - Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, 
  "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", 
                "Насадки удобные, под мебель достаёт легко", 
  ]
}
row_dict = [{"role": "user", "content": f"""{item}"""}]

resp, tokens = await generate_one(row_dict, product_id=item['product_id'])
print('>>> resp\n', resp, tokens)

>>> resp
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером объемом 2 л. Обладает мощностью всасывания 220 Вт и может работать до 60 минут. Весит всего 1,5 кг, что делает его идеальным выбором для бытового использования.",
  "pros": ["Удобные насадки", "Легко подходит под мебель"],
  "cons": ["Щетка забивается волосами", "Чистить неудобно"],
  "tags": ["пылесос", "клин", "бытовая техника", "с контейнером", "220 Вт"]
} 172


In [17]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [23]:
import asyncio, time

async def run_concurrent(items, concurrency, generate_one):
  """Отправляет запросы параллельно, не больше `concurrency` одновременно.

  generate_one — ваша асинхронная функция, принимает один товар и возвращает
  словарь с результатом. Возвращает список результатов и общее время прогона.
  """
  sem = asyncio.Semaphore(concurrency)

  async def wrapped(item):
    async with sem:
      return await generate_one([{"role": "user", "content": item}], json.loads(item)['product_id'])

  started = time.perf_counter()
  results = await asyncio.gather(*[wrapped(it) for it in items])
  return results, time.perf_counter() - started

In [ ]:
import json
from datetime import datetime
from pathlib import Path

def wright_files(timestamp_str_dir, results, params, df):
  timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
  directory = Path(f"./outputs/{timestamp_str_dir}/{timestamp_str}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "results.json", "w", encoding="utf-8") as file:
    file.write(results)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

  totals_path = f"./outputs/{timestamp_str_dir}" / "totals.csv"
  df.to_csv(totals_path, index=False, encoding="utf-8")

In [ ]:
import pandas as pd

its = items[:10]

df = pd.DataFrame(columns=['concurrency', 'total_time', 'time_per_item'])
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

for concurrency in [1, 2, 4, 6]:
# for concurrency in [1]:
  results, time_total = await run_concurrent(its, concurrency, generate_one)
  new_row = {'concurrency': concurrency, 'total_time': time_total, 'time_per_item': round(time_total/len(its), 2)}
  print(new_row)
  # print(results)

  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
  # wright_files(timestamp_str, results, new_row, df)

{'concurrency': 1, 'total_time': 1450.4435011000023, 'time_per_item': 145.04}
{'concurrency': 2, 'total_time': 728.1160350000137, 'time_per_item': 72.81}
{'concurrency': 4, 'total_time': 8129.241237599985, 'time_per_item': 812.92}


C:\Temp\Python\Lib\dis.py:545: RuntimeWarning: coroutine 'AsyncClient.chat' was never awaited
  def get_argval_argrepr(self, op, arg, offset):


CancelledError: 

In [33]:
df_sorted = df.sort_values(by='time_per_item')
df_sorted

,concurrency,temperature,top_p,top_k,repetition_penalty,max_tokens,total_time,time_per_item
1,2,NaN,NaN,NaN,NaN,NaN,28.805129,28.81
2,4,NaN,NaN,NaN,NaN,NaN,30.954601,30.95
3,8,NaN,NaN,NaN,NaN,NaN,31.295515,31.3
0,1,NaN,NaN,NaN,NaN,NaN,31.379664,31.38


In [35]:
totals_path = Path(f"./outputs/totals_final_{timestamp_str}.csv")
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")

In [ ]:
import time
import json

items_slice = items[:]
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
err = 0

params = {
  "temperature":        0.3, 
  "top_p":              0.9, 
  "top_k":              50, 
  "repetition_penalty": 1.1, 
  "max_tokens":         350,
  "products":           len(items_slice), 
}

started_batch = time.perf_counter()

for item in items_slice:
  row_dict = {"role": "user", "content": item}
  item_dict = json.loads(item)

  started = time.perf_counter()
  results, tokens = generate_one([row_dict], product_id=item_dict['product_id'], params=params)
  generation_time = time.perf_counter() - started

  if 'error' in results:
    err += 1
  else:
    card_save(timestamp_str, results, item_dict['product_id'])
    
  new_row = {'card': item_dict['product_id'], 'error': err, 'tokens': tokens, 'time_per_item': round(generation_time, 2)}
  print(new_row)

batch_generation_time = round(time.perf_counter() - started_batch, 2)
params_save(
  timestamp_str, 
  params | 
  {
    "errors": err, 
    "error_percent": round(100 * err / len(items_slice), 2), 
    "total_time": batch_generation_time, 
    'time_per_item': round(batch_generation_time / len(items_slice), 2)
  }
)

{'card': 'P-0027', 'error': 0, 'tokens': 161, 'time_per_item': 34.98}
{'card': 'P-0032', 'error': 0, 'tokens': 164, 'time_per_item': 38.49}
{'card': 'P-0017', 'error': 0, 'tokens': 165, 'time_per_item': 37.83}
{'card': 'P-0021', 'error': 0, 'tokens': 168, 'time_per_item': 62.66}
{'card': 'P-0036', 'error': 0, 'tokens': 186, 'time_per_item': 68.22}
{'card': 'P-0039', 'error': 0, 'tokens': 235, 'time_per_item': 381.32}
{'card': 'P-0024', 'error': 0, 'tokens': 191, 'time_per_item': 72.31}
{'card': 'P-0009', 'error': 0, 'tokens': 145, 'time_per_item': 325.0}
{'card': 'P-0037', 'error': 0, 'tokens': 138, 'time_per_item': 46.49}
{'card': 'P-0002', 'error': 0, 'tokens': 225, 'time_per_item': 351.71}
{'card': 'P-0016', 'error': 0, 'tokens': 170, 'time_per_item': 73.87}
{'card': 'P-0028', 'error': 0, 'tokens': 168, 'time_per_item': 85.2}
{'card': 'P-0012', 'error': 0, 'tokens': 178, 'time_per_item': 70.38}
{'card': 'P-0020', 'error': 0, 'tokens': 160, 'time_per_item': 70.63}
{'card': 'P-0026', 